# EquiTS: Risk-Certified Predictive Quotienting for Multivariate Time-Series
This notebook demonstrates training, evaluating, and benchmarking **EquiTS** and competitive baselines on Google Colab with GPU acceleration.

### Step 1: Clone Repository & Install Dependencies
*(If you already uploaded the folder to Google Drive or Colab, just `%cd` into the directory)*

In [ ]:
# Replace with your GitHub repository URL if cloned from GitHub:
!git clone https://github.com/sh00von/EquiTS.git
%cd EquiTS

!pip install -r requirements.txt

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### Step 2: Run Formal Verification Tests
Verify that the architecture, RevIN, information divergence, and zero-leakage constraints pass.

In [ ]:
!python -m pytest tests -v

### Step 3: Train EquiTS on ETTh1 (Our Model)
Trains EquiTS (Acc-Mode and RCPQ-Memory variants) with the Residual Linear Highway.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Main table: all baselines + EquiTS on ETTh1, TSLib splits, L=96, H=96, seed 2021
!python run.py --dataset ETTh1 --model all --horizon 96 --epochs 10 --seeds 2021 --device {device}

### Step 3b: Train EquiTS on Weather Benchmark (21 Channels)
Evaluates generalization on continuous real-world meteorological dynamics.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# EquiTS on ETTh1 over all horizons (H = 96, 192, 336, 720), seed 2021
!python run.py --dataset ETTh1 --model EquiTS --horizon all --epochs 10 --seeds 2021 --device {device}

### Step 4: Display Benchmark Results Table

In [ ]:
import glob
from IPython.display import Markdown
for path in sorted(glob.glob('results/benchmark_*.md')):
    display(Markdown(open(path).read()))

### Step 5: Evaluate the Pareto Frontier (Speedup vs. Accuracy)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Risk / accuracy / latency trade-off (reuses the seed-2021 checkpoint if it exists)
!python experiments/pareto_sweep.py --dataset ETTh1 --horizon 96 --seed 2021 --device {device}
from IPython.display import Markdown
display(Markdown(open('results/pareto_etth1_H96.md').read()))